# 💧 AquaVigía Colombia — Reentrenamiento Limpio: CNN (MobileNetV2) + Híbrido YOLOv8 → ML

Este notebook resuelve los 3 problemas detectados en el modelo anterior:
1. **Limpieza automática de imágenes basura** (capturas de pantalla con texto o íconos como `IMG_6076.PNG`) y soporte completo de **`.HEIC` + `.JPG` + `.PNG`** con `pillow-heif`.
2. **Opción A — Red Neuronal Convolucional (Transfer Learning con MobileNetV2)**: Aprende patrones visuales reales (formas larvarias, turbidez, verdín) con *Data Augmentation* y exporta a **ONNX / TensorFlow.js** sin errores de `np.object`.
3. **Opción B — Híbrido YOLOv8 → ML**: Usa **YOLOv8** para detectar cajas de larvas y extrae `[num_cajas, conf_max, conf_media, area_total, elongacion_max]` para alimentar un clasificador ML (`RandomForest` / `GradientBoosting`).

In [ ]:
!pip install -q pillow-heif ultralytics tensorflow tf2onnx onnxruntime scikit-learn
import os, glob, json, cv2, numpy as np
from PIL import Image
from pillow_heif import register_heif_opener
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

register_heif_opener()
print('TensorFlow:', tf.__version__, '| HEIC habilitado')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Imagenes de larbas'
RUTA_SIN_RIESGO = os.path.join(BASE_DIR, 'sin-factor_de_riesgo')
RUTA_CON_LARVA  = os.path.join(BASE_DIR, 'factor_de_riesgo', 'con-larva')
RUTA_SIN_LARVA  = os.path.join(BASE_DIR, 'factor_de_riesgo', 'sin_larva')

# Lista explícita de capturas/basura a excluir del entrenamiento
EXCLUDED_FILES = {'IMG_6076.PNG'}

def es_imagen_basura(img_bgr, filename):
    if filename in EXCLUDED_FILES:
        return True, 'En lista de exclusión (captura con texto)'
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # Si más del 45% de los píxeles caen en un solo bloque de color plano, suele ser captura/gráfico
    hist = cv2.calcHist([gray], [0], None, [16], [0, 256]).flatten() / gray.size
    if np.max(hist) > 0.55:
        return True, f'Fondo plano artificial ({np.max(hist):.2f})'
    return False, 'OK'

def cargar_imagen(fp):
    if fp.lower().endswith('.heic'):
        return cv2.cvtColor(np.array(Image.open(fp).convert('RGB')), cv2.COLOR_RGB2BGR)
    return cv2.imread(fp)

X_imgs, y_risk, y_larva = [], [], []
IMG_SIZE = (128, 128)
EXTS = ('.jpg', '.jpeg', '.png', '.heic')

for folder, r_lbl, l_lbl in [(RUTA_SIN_RIESGO, 0, 0), (RUTA_SIN_LARVA, 1, 0), (RUTA_CON_LARVA, 1, 1)]:
    if not os.path.exists(folder): continue
    descartadas = 0
    for fn in os.listdir(folder):
        if not fn.lower().endswith(EXTS): continue
        img = cargar_imagen(os.path.join(folder, fn))
        if img is None: continue
        basura, motivo = es_imagen_basura(img, fn)
        if basura:
            descartadas += 1
            continue
        rgb = cv2.cvtColor(cv2.resize(img, IMG_SIZE), cv2.COLOR_BGR2RGB)
        X_imgs.append(rgb)
        y_risk.append(r_lbl)
        y_larva.append(l_lbl)
    print(f'{os.path.basename(folder)}: cargadas={sum(1 for r in y_risk if r==r_lbl)}, descartadas={descartadas}')

X_imgs = np.array(X_imgs, dtype=np.float32)
y_risk = np.array(y_risk)
y_larva = np.array(y_larva)

In [ ]:
# OPCIÓN A: Transfer Learning con MobileNetV2 (Etapa 1 y Etapa 2)
def construir_mobilenet_binario(input_shape=(128, 128, 3)):
    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal_and_vertical'),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomContrast(0.15),
    ])
    base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base.trainable = False  # Congelar extractor convolucional pre-entrenado
    inp = tf.keras.Input(shape=input_shape)
    x = aug(inp)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return model

X_tr, X_val, yr_tr, yr_val = train_test_split(X_imgs, y_risk, test_size=0.2, stratify=y_risk, random_state=42)
cnn_riesgo = construir_mobilenet_binario()
cnn_riesgo.fit(X_tr, yr_tr, validation_data=(X_val, yr_val), epochs=15, batch_size=16)
cnn_riesgo.save('mobilenet_etapa1_riesgo.keras')

# Etapa 2 solo sobre imágenes con factor de riesgo
mask_r = (y_risk == 1)
Xl_tr, Xl_val, yl_tr, yl_val = train_test_split(X_imgs[mask_r], y_larva[mask_r], test_size=0.2, stratify=y_larva[mask_r], random_state=42)
cnn_larva = construir_mobilenet_binario()
cnn_larva.fit(Xl_tr, yl_tr, validation_data=(Xl_val, yl_val), epochs=18, batch_size=16)
cnn_larva.save('mobilenet_etapa2_larva.keras')

# Exportar a ONNX (compatible con ONNX Runtime Web en Netlify sin error de np.object)
!python3 -m tf2onnx.convert --keras mobilenet_etapa1_riesgo.keras --output mobilenet_etapa1_riesgo.onnx
!python3 -m tf2onnx.convert --keras mobilenet_etapa2_larva.keras --output mobilenet_etapa2_larva.onnx

In [ ]:
# OPCIÓN B: Híbrido YOLOv8 -> ML (Las cajas de YOLO alimentan al modelo de ML)
from ultralytics import YOLO

# Carga tu modelo YOLOv8 entrenado (best.pt) sobre el dataset anotado
# yolo_model = YOLO('runs/detect/aquavigia_yolov8_larvas/weights/best.pt')
def extraer_features_yolo_para_ml(yolo_model, img_bgr):
    res = yolo_model.predict(img_bgr, conf=0.25, verbose=False)[0]
    boxes = res.boxes
    if len(boxes) == 0:
        return [0, 0.0, 0.0, 0.0, 0.0]
    confs = boxes.conf.cpu().numpy()
    xywhn = boxes.xywhn.cpu().numpy()
    areas = xywhn[:, 2] * xywhn[:, 3]
    aspects = np.maximum(xywhn[:, 2], xywhn[:, 3]) / np.maximum(1e-4, np.minimum(xywhn[:, 2], xywhn[:, 3]))
    return [len(boxes), float(np.max(confs)), float(np.mean(confs)), float(np.sum(areas)), float(np.max(aspects))]

print('Función híbrida YOLOv8 -> ML lista para concatenar con features HSV/textura y entrenar RandomForestClassifier.')